# 5. Genetic Algorithms with SeqTO-v1

## 5.1 Why a Genetic Algorithm for Topology Optimisation?

Topology optimisation of electromagnetic devices has three properties that make it hard for classical optimisers:

1. **Discrete design variables.** The decision at each cell is binary (iron / air); there is no continuous gradient to follow.
2. **Black-box, FEA-evaluated objective.** Each candidate design requires a finite-element solve; the cost-per-evaluation is the dominant resource constraint.
3. **Non-convex, highly multimodal landscape.** A small mask change can reroute flux entirely, producing large discontinuous jumps in the reward.

These properties have made **genetic algorithms** the dominant prior-art optimiser for discrete EM topology problems for three decades {cite}`zhao2016topology_ga_synrm,goldberg1989genetic,holland1992adaptation`. A GA maintains a population of candidate topologies, applies selection / recombination / mutation each generation, and evaluates each individual with a single FEA call.

The SeqTO-v1 reformulation strengthens the GA / TO combination in one specific way: because the chromosome is an **action sequence** that drives a deposition controller (Section 4), every individual produced by selection, crossover, or mutation is automatically a **connected, manufacturable topology**. ON/OFF GAs, by contrast, must add a connectivity penalty $\lambda_c (N_{\text{regions}} - 1)$ to the fitness — adding a hyperparameter to tune and wasting evaluations on disconnected designs.

Two implementations live in this directory:

- [`ga_cCore.py`](ga_cCore.py) — GA for the **18 × 35 Cartesian C-core** environment (`seqTO_env.py`).
- [`ga_synrm.py`](ga_synrm.py) — GA for the **5 × 5 polar SynRM** environment (`seqTO_SynRM_env.py`).

Both are written in pure Python on top of [DEAP](https://deap.readthedocs.io/) and call each machine's own `calculate_reward()` — the same scalar consumed downstream by the Q-learning agent.

## 5.2 Genetic Algorithm Foundations

This section reviews the algorithmic ingredients needed to read the rest of the chapter without prior GA background. Readers already comfortable with GAs can skip to Section 5.3.

### 5.2.1 Why a Population-Based, Stochastic Search?

Most engineering optimisation works on continuous, differentiable objectives where gradients point reliably toward improvement. Topology optimisation is the opposite case: the design variable is a *discrete* iron / air assignment for each cell, FEA is a black-box mapping with no usable gradient, and small mask changes can cause large reward jumps as flux paths reroute. Two consequences follow:

- A single-trajectory hill-climber gets trapped in the first local optimum the FEA reward function presents.
- We have no gradient to climb out of that basin, so we have to *explore*.

Genetic Algorithms address both problems with the same mechanism: maintain a **population** of candidate solutions in parallel, let them recombine and mutate, and keep the fittest. A population diverse enough to span multiple basins lets the better basin gradually take over as worse individuals are selected away.

### 5.2.2 Vocabulary

| Term | Meaning in SeqTO-v1 |
|---|---|
| *Individual / chromosome* | One candidate action sequence $\mathbf{a} = [a_1, \ldots, a_m]$ |
| *Gene* | One action $a_i \in \{0, 1, 2, 3\}$ |
| *Population* | A list of $N$ individuals evaluated at the same generation |
| *Generation* | One full *select → mate → mutate → evaluate* cycle |
| *Fitness* | Scalar from $f(\mathbf{a}) = \text{FEA}\bigl(\tau(\mathbf{a})\bigr)$ — mean $\lvert B \rvert$ for C-core, max $\lvert \text{torque} \rvert$ for SynRM |
| *Elitism* | Preserving the top-$k$ individuals unchanged across generations |

### 5.2.3 Selection

Selection chooses which individuals survive to mate. Two common operators:

- **Tournament selection** (used throughout this chapter). Pick $k$ random individuals (here $k = 3$); the one with the highest fitness wins and is copied into the mating pool. Repeat $N - \text{elite}$ times to fill the pool. Tournament selection gives a tunable *selection pressure*: larger $k$ → more aggressive convergence, narrower exploration.
- **Roulette-wheel (fitness-proportional)**. Probability of being chosen is $f_i / \sum_j f_j$. Simple but breaks when fitnesses are zero (our budget-violating individuals get fitness zero, so roulette would degenerate). Avoided here.

### 5.2.4 Crossover

Crossover recombines two selected parents into two offspring. For *ordered* chromosomes like ours — where the position of an action matters because every gene shifts the entire downstream trajectory — three crossover operators dominate:

- **Single-point**: cut both parents at the same random position $k$ and swap tails.
- **Two-point** (`tools.cxTwoPoint`, used here): cut at two positions $k_1 < k_2$ and swap the middle segment. Preserves both head and tail of each parent more often than single-point, mitigating *linkage disruption* when adjacent genes encode a related sub-structure.
- **Uniform**: each gene independently inherited from either parent with probability $\tfrac{1}{2}$. Heavy disruption — useful for unstructured chromosomes but a poor fit for action sequences.

The crossover probability $p_{\text{cx}}$ controls how aggressively the pool recombines (here $0.8$ per pair). Higher values increase mixing at the cost of slower fixation on good building blocks.

### 5.2.5 Mutation

Mutation injects fresh genetic material. Without it, after a few generations the population has converged to copies of the few elites and search stalls.

- For integer chromosomes, `tools.mutUniformInt(low=0, up=3, indpb=…)` independently replaces each gene with a random value in $\{0, 1, 2, 3\}$ with per-gene probability $p_{\text{ind}}$.
- A per-gene rate of $p_{\text{ind}} \approx 1/m$ (where $m$ is chromosome length) gives roughly one mutation per individual on average — a common heuristic.
- The per-individual probability $p_{\text{mut}}$ (here $0.2$–$0.3$) gates whether mutation is applied to a given offspring at all.

Mutation rate too low → *premature convergence* (population locked on a local optimum). Too high → search behaves like random restart and never builds on improvements. The $1/m$ default is defensible but ultimately tuned against convergence diagnostics.

### 5.2.6 Elitism and the Hall-of-Fame

Pure tournament + crossover + mutation can stochastically *lose* the best-so-far individual to a bad selection draw or a destructive mutation. **Elitism** guards against that: the top-$k$ individuals (the *Hall of Fame*) are copied into every new generation unchanged. This guarantees the best-so-far fitness is monotone non-decreasing across generations, which makes convergence detection meaningful.

### 5.2.7 Termination by Stall

Both implementations terminate when the best fitness has not improved for $S$ consecutive generations (the `stall_limit` parameter — $25$ for C-core, $12$ for SynRM). This adapts to problem difficulty: easy problems exit early, hard ones run longer. The choice of $S$ trades cost against false-positive convergence — too small and the GA quits before climbing out of a plateau, too large and you burn FEMM evaluations on a population that has already converged.

### 5.2.8 Why a GA for this Problem?

Three reasons GA is a strong default for SeqTO-v1:

1. **Discrete, gradient-free, expensive evaluation.** FEMM solves cost ~100 ms (C-core) to ~60 s (SynRM with 6-angle sweep). GA's parallel-population structure amortises this cost, and unlike SIMP / level-set methods each evaluation receives a *complete* topology rather than a small perturbation — so every FEA call returns a usable signal.
2. **No connectivity penalty needed.** The SeqTO chromosome guarantees connected topologies by construction (Section 5.1). In an ON/OFF GA formulation we would have to add and tune a $\lambda_c (N_{\text{regions}} - 1)$ penalty term, and waste evaluations on disconnected designs.
3. **Solid empirical track record.** GAs have dominated discrete electromagnetic topology optimisation for three decades {cite}`zhao2016topology_ga_synrm`. Using GA as the gradient-free baseline puts the SeqTO-v1 reformulation in direct comparison with the dominant prior-art algorithm class.

## 5.3 GA Formulation for SeqTO-v1

### 5.3.1 Chromosome Encoding

In SeqTO-v1, the optimisation variable is an **action sequence** — not a cell-by-cell material assignment. Each individual (chromosome) in the GA population is an ordered list of $m$ directional moves:

$$\mathbf{a} = [a_1, a_2, \ldots, a_m], \quad a_i \in \{0, 1, 2, 3\}$$

where the action alphabet for C-core matches `seqTO_env.py` and `ga_cCore.py` exactly:

| code | action | $(\Delta r, \Delta c)$ |
|---|---|---|
| 0 | RIGHT | $(0, +1)$ |
| 1 | LEFT  | $(0, -1)$ |
| 2 | UP    | $(-1, 0)$ |
| 3 | DOWN  | $(+1, 0)$ |

The topology $\tau(\mathbf{a})$ is generated deterministically by executing the sequence with the $3 \times 3$ deposition controller — every position the controller passes through deposits a $3 \times 3$ patch of iron (subject to the design-window and right-coil exclusion masks defined in Section 4). The SynRM variant uses the same four-symbol alphabet but with **polar** semantics (UP = radially outward, RIGHT = increasing angular sector); see Section 5.7.

**Contrast with ON/OFF encoding:**
- ON/OFF GA chromosome: $\mathbf{c} \in \{0, 1\}^n$ — $n$ independent binary cell assignments (disconnected topologies possible)
- SeqTO-v1 GA chromosome: $\mathbf{a} \in \{0, 1, 2, 3\}^m$ — $m$ ordered actions ($m \ll n$, connectivity guaranteed)

For the C-core experiment the design window contains $n = 288$ designable cells, while the action sequence has length $m = 34$. The search-space dimensionality is therefore reduced from $2^{288}$ (ON/OFF) to $4^{34}$ — still large, but on a structurally connected manifold.

### 5.3.2 Fitness Function

The fitness of a chromosome is the FEMM-evaluated objective of the topology it generates. For C-core:

$$f(\mathbf{a}) = \bigl(\tfrac{1}{6} \sum_{p=1}^{6} \lvert B_p \rvert \bigr) \times 1000$$

where $\lvert B_p \rvert$ is the flux-density magnitude at six fixed sample points inside the armature — the **same scalar** that `seqTO_env.calculate_reward()` returns. The mean-$\lvert B \rvert$ proxy is used during search; conversion to force in Newtons is a separate post-hoc Maxwell-stress-tensor calculation deferred to a later pass.

The SynRM fitness (Section 5.7) uses a different scalar — max $\lvert \text{torque} \rvert$ across 6 rotor angles — but the framework is identical.

**Budget constraint.** Iron-cell counts outside $[K_1, K_2]$ are mapped to a fitness of zero **before** the FEMM call is even made. This (i) enforces the manufacturing-budget constraint as a hard filter, (ii) skips wasted FEMM solves on out-of-budget topologies, and (iii) keeps the early generations' all-zero rows in the logbook honest about how much of the random initial population is infeasible.

**No connectivity penalty is needed** because $\tau(\mathbf{a})$ is always a connected material distribution by construction.

### 5.3.3 Genetic Operators (DEAP)

| Role | DEAP primitive | Parameter |
|---|---|---|
| Selection | `tools.selTournament` | `tournsize = 3` |
| Crossover | `tools.cxTwoPoint` | $p_{\text{cx}} = 0.8$ per pair |
| Mutation | `tools.mutUniformInt(low=0, up=3)` | $p_{\text{ind}} = \max(0.02, 1/m)$ per gene; $p_{\text{mut}} = 0.2$ per individual |
| Elitism | `tools.HallOfFame` | size 5 (C-core), 3 (SynRM); re-injected each generation |
| Termination | stall counter | stop after $S = 25$ (C-core) / $12$ (SynRM) generations without improvement |

Crossover (`cxTwoPoint`) recombines topological sub-structures: a flux-return path from parent 1 can be glued onto an armature-approach path from parent 2. Mutation is per-gene uniform-integer on $\{0, 1, 2, 3\}$; because actions are ordered and each deposits a spatially coherent block, a single gene change shifts the entire downstream trajectory.

## 5.4 Topology Decoding and Population Seeding (C-core)

Two design choices in `ga_cCore.py` have a large empirical effect on early-generation feasibility:

1. **Biased action sampling.** Random individuals are not drawn uniformly from $\{0, 1, 2, 3\}$. RIGHT and DOWN are weighted at 35 % each, LEFT and UP at 15 % each. This biases random walks toward the natural diagonal of the design window (which the controller enters from the top-left) and reduces the fraction of trajectories that backtrack out of the design window.

2. **Boustrophedon (snake-sweep) seed.** A single deterministic individual that sweeps left–right–down across the design window is *always* inserted into generation 0. In a clear design window this seed would deposit $\sim 80$ cells, but in the C-core's actual design window the right-sweep gets stuck on the right-coil exclusion zone at column 11 (the `in_coil2` guard rejects moves into rows 0–8 × cols 11–13). The result — visible in the cell output below — is that the seed lands at only $\sim 33$ cells, well below $K_1 = 80$. Consequently gen-0 max fitness is exactly $0$ (the logbook in Section 5.6 confirms this); feasibility only emerges in gen 3 once crossover / mutation produce an action sequence that escapes the coil trap.

The cell below is a runnable version of the pure-Python decoder + seed generator (a slimmed copy of `ga_cCore.py`), ending with a visualisation of the boustrophedon seed's iron-mask so you can see exactly how the COIL2 exclusion limits its coverage:

In [ ]:
# C-core SeqTO-v1 topology decoder — pure Python, no FEMM required.
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

# Geometry constants (must match seqTO_env.py)
ROWS, COLS = 18, 35
COIL_LEN   = 9
COIL1_COLS = (2, 5)
COIL2_COLS = (11, 14)
ARM_ROWS, ARM_COLS = (0, 15), (27, 33)
DW_ROWS, DW_COLS   = (0, 15), (5, 26)
K1, K2     = 80, 180
ACTIONS    = {0: ('RIGHT', 0, +1), 1: ('LEFT', 0, -1),
              2: ('UP',   -1,  0), 3: ('DOWN', +1,  0)}


def is_design_cell(i, j):
    in_dw    = DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]
    in_coil2 = i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]
    return in_dw and not in_coil2


def build_topology(action_seq):
    """Execute the action sequence with a 3x3 deposition brush; return iron_mask."""
    iron_mask = np.zeros((ROWS, COLS), dtype=int)
    pos_r, pos_c = DW_ROWS[0] + 1, DW_COLS[0] + 1
    for action in action_seq:
        _, dr, dc = ACTIONS[int(action)]
        new_r, new_c = pos_r + dr, pos_c + dc
        in_dw    = DW_ROWS[0] <= new_r < DW_ROWS[1] and DW_COLS[0] <= new_c < DW_COLS[1]
        in_coil2 = new_r < COIL_LEN and COIL2_COLS[0] <= new_c < COIL2_COLS[1]
        if in_dw and not in_coil2:
            pos_r, pos_c = new_r, new_c
            for di in range(-1, 2):
                for dj in range(-1, 2):
                    ri, ci = pos_r + di, pos_c + dj
                    if is_design_cell(ri, ci):
                        iron_mask[ri, ci] = 1
    return iron_mask


def make_sweep_individual(seq_len):
    """Boustrophedon (snake) sweep — a deterministic seed that reliably deposits >= 80 cells."""
    actions, going_right = [], True
    while len(actions) < seq_len:
        for _ in range(16):
            actions.append(0 if going_right else 1)
            if len(actions) >= seq_len: break
        going_right = not going_right
        for _ in range(3):
            actions.append(3)
            if len(actions) >= seq_len: break
    return actions[:seq_len]


def plot_ccore(mask, title):
    """Render a C-core mask alongside coil/armature regions."""
    region = np.zeros((ROWS, COLS), dtype=int)
    for i in range(ROWS):
        for j in range(COLS):
            if i < COIL_LEN and COIL1_COLS[0] <= j < COIL1_COLS[1]:        region[i, j] = 1
            elif i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]:      region[i, j] = 2
            elif ARM_ROWS[0] <= i < ARM_ROWS[1] and ARM_COLS[0] <= j < ARM_COLS[1]: region[i, j] = 3
            elif is_design_cell(i, j):                                      region[i, j] = 4 if mask[i, j] else 5
            elif DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]:   region[i, j] = 6
    CLRS = ['#D5D8DC', '#2980B9', '#E67E22', '#27AE60', '#2C3E50', '#FEF9E7', '#FAD7A0']
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.imshow(region, cmap=ListedColormap(CLRS), vmin=0, vmax=6, origin='upper', aspect='equal')
    ax.set_title(title)
    ax.set_xlabel('column'); ax.set_ylabel('row')
    ax.legend(handles=[
        mpatches.Patch(fc='#2C3E50', label='Iron (deposited)'),
        mpatches.Patch(fc='#FEF9E7', ec='grey', lw=0.5, label='Air (design window)'),
        mpatches.Patch(fc='#2980B9', label='Left coil'),
        mpatches.Patch(fc='#E67E22', label='Right coil'),
        mpatches.Patch(fc='#27AE60', label='Armature'),
    ], fontsize=8, loc='upper right')
    plt.tight_layout()
    plt.show()


# Demonstrate: build the boustrophedon seed and visualise
sweep = make_sweep_individual(34)
mask  = build_topology(sweep)
plot_ccore(mask, f'C-core boustrophedon seed — {int(mask.sum())} iron cells')

print(f"Sweep deposited {int(mask.sum())} iron cells out of 288 designable.")
print(f"Budget K1={K1}, K2={K2}: {'IN' if K1 <= int(mask.sum()) <= K2 else 'OUT OF'} range.")

## 5.5 DEAP Setup and Fitness Evaluation

The GA implementation uses [DEAP](https://deap.readthedocs.io/) (Distributed Evolutionary Algorithms in Python). The full toolbox registration lives in `ga_cCore.py:_setup_deap()` (and the SynRM equivalent in `ga_synrm.py`); both files register the same five primitives — see the table in Section 5.3.3.

The fitness function applies the budget filter (hard-zero for out-of-budget topologies) **before** invoking the FEMM evaluation, so no FEA call is wasted on infeasible chromosomes:

```python
def evaluate(individual):
    iron_mask  = build_topology(individual)
    iron_count = int(iron_mask.sum())
    if not (K1 <= iron_count <= K2):
        return (0.0,)
    from seqTO_env import calculate_reward     # lazy — needs pyfemm + FEMM 4.2
    reward, _, _ = calculate_reward(iron_mask)
    return (reward,)
```

The `from … import calculate_reward` is inside the function so the GA module is importable on machines without `pyfemm`. The DEAP toolbox is registered once per run by `_setup_deap()`:

```python
tb.register('mate',    tools.cxTwoPoint)
tb.register('mutate',  tools.mutUniformInt, low=0, up=3,
            indpb=max(0.02, 1.0 / seq_len))
tb.register('select',  tools.selTournament, tournsize=3)
```

The complete source is in `ga_cCore.py` (lines 96–166) and `ga_synrm.py` (lines 107–170).

### 5.5.1 Experimental Configuration (C-core)

Defaults match the CLI defaults of `ga_cCore.py` (`python ga_cCore.py` with no flags reproduces the run below):

| Parameter | Value | Source |
|---|---|---|
| Chromosome length $m$ | 34 | matches Q-learning `max_steps` |
| Population size | 50 | `--pop` |
| Max generations | 200 | `--n-gen` |
| Crossover prob. $p_{\text{cx}}$ | 0.8 per pair | `--cxpb` |
| Mutation prob. $p_{\text{mut}}$ | 0.2 per individual | `--mutpb` |
| Per-gene mutation $p_{\text{ind}}$ | $\max(0.02,\, 1/m) \approx 0.029$ | `mutUniformInt` |
| Hall-of-Fame | 5 elites | `--hof` |
| Stall termination | 25 generations | `--stall` |
| Iron budget $[K_1, K_2]$ | $[80, 180]$ cells | hard-zero outside |
| Random seed | 42 | `--seed` |

Each generation evaluates only individuals whose fitness has been invalidated by crossover or mutation, plus the elite re-injection from the Hall-of-Fame. Out-of-budget individuals consume zero FEMM evaluations.

## 5.6 Run Log and Convergence (C-core)

Running `python ga_cCore.py` with the defaults above produces the logbook at `designs/ga/ga_logbook.json` and the best mask at `designs/ga/ga_best_mask.npy`. Summary of that run:

| Metric | Value |
|---|---|
| Generations executed | 48 (terminated by stall criterion) |
| Total FEMM evaluations | 1,808 |
| First non-zero fitness | gen 3 ($f = 1{,}161.4$) |
| Global best fitness | $f^\star = 1{,}628.61$ at gen 22 |
| Iron-cell count of best | 98 cells (well inside $[80, 180]$) |
| Plateau length | 25 generations with no improvement after gen 22 |

Behavioural notes:

- *Generations 0–2 are all zero.* Every individual in the initial population is infeasible, **including the boustrophedon seed** — see Section 5.4 for the COIL2-trap analysis (the seed lands at $\sim 33$ cells, below $K_1 = 80$). Feasibility only emerges in gen 3 once mutation / crossover produce a chromosome whose trajectory escapes the right-coil exclusion.
- *Rapid improvement gen 3 → 13.* Best fitness rises from 1,161 to 1,521 as the first feasible offspring proliferates through tournament selection.
- *Slow refinement gen 14 → 22.* From 1,554 to 1,628.61. The mean fitness tracks the best within ~1 % by gen 30, indicating the population has converged onto the basin of the eventual optimum.
- *Stall termination.* No further improvement is found in generations 23–47, and the stall counter (limit 25) cuts the run at gen 47.

The cell below loads the saved logbook + best-mask and renders the two-panel result figure inline:

In [ ]:
# Visualise the C-core GA run from saved artifacts in designs/ga/
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap

lb        = json.load(open('designs/ga/ga_logbook.json'))
best_mask = np.load('designs/ga/ga_best_mask.npy')

gens   = [r['gen']  for r in lb]
maxfit = [r['max']  for r in lb]
mfit   = [r['mean'] for r in lb]
mnfit  = [r['min']  for r in lb]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Convergence curve
ax = axes[0]
ax.plot(gens, maxfit, 'crimson',   lw=2,   label='Best')
ax.plot(gens, mfit,   'steelblue', lw=1.5, label='Mean')
ax.fill_between(gens, mnfit, maxfit, alpha=0.15, color='steelblue')
ax.set_xlabel('Generation')
ax.set_ylabel('Fitness  (mean |B| × 1000)')
ax.set_title('C-core GA convergence')
ax.legend(); ax.grid(alpha=0.3)

# Best-topology region map (reuse logic from the helpers cell above)
region = np.zeros((ROWS, COLS), dtype=int)
for i in range(ROWS):
    for j in range(COLS):
        if i < COIL_LEN and COIL1_COLS[0] <= j < COIL1_COLS[1]:    region[i, j] = 1
        elif i < COIL_LEN and COIL2_COLS[0] <= j < COIL2_COLS[1]:  region[i, j] = 2
        elif ARM_ROWS[0] <= i < ARM_ROWS[1] and ARM_COLS[0] <= j < ARM_COLS[1]: region[i, j] = 3
        elif is_design_cell(i, j):                                  region[i, j] = 4 if best_mask[i, j] else 5
        elif DW_ROWS[0] <= i < DW_ROWS[1] and DW_COLS[0] <= j < DW_COLS[1]: region[i, j] = 6

CLRS = ['#D5D8DC', '#2980B9', '#E67E22', '#27AE60', '#2C3E50', '#FEF9E7', '#FAD7A0']
ax = axes[1]
ax.imshow(region, cmap=ListedColormap(CLRS), vmin=0, vmax=6, origin='upper', aspect='equal')
ax.set_title(f'GA best topology — {int(best_mask.sum())} iron cells, fitness = {max(maxfit):.2f}')
ax.set_xlabel('column'); ax.set_ylabel('row')
ax.legend(handles=[
    mpatches.Patch(fc='#2C3E50', label='Iron (GA)'),
    mpatches.Patch(fc='#FEF9E7', ec='grey', lw=0.5, label='Air'),
    mpatches.Patch(fc='#2980B9', label='Left coil'),
    mpatches.Patch(fc='#E67E22', label='Right coil'),
    mpatches.Patch(fc='#27AE60', label='Armature'),
], fontsize=8, loc='lower right')

plt.suptitle('C-core SeqTO-v1 — GA Result (DEAP)', fontsize=12)
plt.tight_layout()
plt.show()

best_gen = gens[maxfit.index(max(maxfit))]
print(f"Best fitness  : {max(maxfit):.4f}  at gen {best_gen}")
print(f"Iron cells    : {int(best_mask.sum())} / 288 designable  (budget {K1}..{K2})")
print(f"Total gens    : {len(lb)}  (stall-terminated)")
print(f"Total FEMM ev.: {sum(r['nevals'] for r in lb)}")

## 5.7 Application to the SynRM Environment

The same chromosome formulation transfers to the **synchronous reluctance motor** (SynRM) environment of `seqTO_SynRM_env.py` with only a handful of changes — none of which touch the GA algorithm itself, only the topology decoder.

| Aspect | C-core | SynRM |
|---|---|---|
| Design grid | $18 \times 35$ Cartesian, 288 designable cells | $5 \times 5$ polar (radial × angular), 25 cells |
| Controller | $3 \times 3$ brush, deposits a $3 \times 3$ block per step | $1 \times 1$ brush, deposits a single cell |
| Action semantics | RIGHT/LEFT = Cartesian $\pm$col; UP/DOWN = $\mp$row | RIGHT/LEFT = angular index $j \pm 1$; UP/DOWN = radial index $i \pm 1$ (UP = outward) |
| Iron budget | $80 \le \text{iron} \le 180$ | $5 \le \text{iron} \le 20$ |
| Fitness | mean $\lvert B \rvert$ at 6 armature points × 1000 | max $\lvert \text{torque} \rvert$ across 6 rotor angles |
| Chromosome length | 34 actions | 15 actions (= MDP `MAX_STEPS`) |
| Search-space size | $4^{34} \approx 3 \times 10^{20}$ | $4^{15} \approx 10^9$ |
| Cost per evaluation | 1 FEMM solve, ~100 ms | 6 FEMM solves (rotor-angle sweep), ~60 s |

The polar action semantics deserve a closer look: a `RIGHT` action increments the angular index $j$ from 0 to 4 (covering 0° → 90° in 18° sectors), and an `UP` action increments the radial index $i$ from 0 to 4 (covering $R_{\text{HUB}}$ → $R_{\text{PERI\_IN}}$). The controller starts at $(i, j) = (0, 0)$ — innermost radial band, most-CW angular sector — and deposits exactly one $1 \times 1$ cell per step at the position the controller occupies *after* the move attempt. Out-of-grid moves are silently rejected; the controller stays in place but still deposits, which matches the production `SynRMSeqTOEnv.step()` behaviour exactly.

### 5.7.1 Pure-Python topology decoder

The decoder (verbatim logic from `ga_synrm.py`) is much shorter than the C-core's because there is no flux-channel exclusion and no $3 \times 3$ brush. The cell below defines it, runs the boustrophedon seed, and renders the result in polar coordinates:

In [ ]:
# SynRM SeqTO-v1 topology decoder — pure Python, no FEMM required.
import numpy as np
import matplotlib.pyplot as plt

# Polar geometry constants (match seqTO_SynRM_env.py)
DD_ROWS_S, DD_COLS_S = 5, 5
R_HUB, R_PERI_IN     = 12.0, 36.7
DR_S    = (R_PERI_IN - R_HUB) / DD_ROWS_S      # 4.94 mm per band
DTHETA_S = 90.0 / DD_COLS_S                    # 18 deg per sector
K1_S, K2_S = 5, 20
ACTIONS_S  = {0: ('RIGHT', 0, +1), 1: ('LEFT', 0, -1),
              2: ('UP',   +1,  0), 3: ('DOWN', -1,  0)}


def is_in_grid(i, j):
    return 0 <= i < DD_ROWS_S and 0 <= j < DD_COLS_S


def build_topology_synrm(action_seq):
    """Mirror of SynRMSeqTOEnv.step() — 1x1 brush, always deposits at current pos."""
    iron_mask = np.zeros((DD_ROWS_S, DD_COLS_S), dtype=int)
    pos_r, pos_c = 0, 0
    for action in action_seq:
        _, dr, dc = ACTIONS_S[int(action)]
        new_r, new_c = pos_r + dr, pos_c + dc
        if is_in_grid(new_r, new_c):
            pos_r, pos_c = new_r, new_c
        iron_mask[pos_r, pos_c] = 1
    return iron_mask


def make_sweep_synrm(seq_len):
    """Boustrophedon seed for the 5x5 polar grid."""
    actions, going_right = [], True
    while len(actions) < seq_len:
        for _ in range(DD_COLS_S - 1):
            actions.append(0 if going_right else 1)
            if len(actions) >= seq_len: break
        if len(actions) >= seq_len: break
        going_right = not going_right
        actions.append(2)
    return actions[:seq_len]


def plot_polar(mask, title):
    """Polar visualisation of a 5x5 SynRM rotor mask."""
    fig, ax = plt.subplots(figsize=(5.5, 5.5), subplot_kw={'projection': 'polar'})
    ax.set_theta_zero_location('E'); ax.set_theta_direction(1)
    ax.set_thetamin(0); ax.set_thetamax(90)
    ax.set_ylim(0, R_PERI_IN + 4)
    n_arc = 16
    for i in range(DD_ROWS_S):
        for j in range(DD_COLS_S):
            r1 = R_HUB + i * DR_S
            r2 = R_HUB + (i + 1) * DR_S
            t1 = np.radians(j * DTHETA_S)
            t2 = np.radians((j + 1) * DTHETA_S)
            theta_arc  = np.linspace(t1, t2, n_arc)
            theta_poly = np.concatenate([theta_arc, theta_arc[::-1]])
            r_poly     = np.concatenate([np.full(n_arc, r1), np.full(n_arc, r2)])
            color = '#2C3E50' if mask[i, j] else '#FEF9E7'
            ax.fill(theta_poly, r_poly, color=color, edgecolor='#7F8C8D', linewidth=0.4)
    ax.plot([np.radians(45)] * 2, [0, R_PERI_IN + 2], 'r--', lw=1.0, alpha=0.7)
    ax.text(np.radians(45), R_PERI_IN + 3.5, 'd-axis (geometric)',
            ha='center', va='bottom', fontsize=8, color='red')
    ax.set_title(f'{title}\niron cells = {int(mask.sum())} / 25', pad=18)
    ax.set_yticks([R_HUB, R_PERI_IN])
    ax.set_yticklabels([f'{R_HUB:.1f}', f'{R_PERI_IN:.1f}'], fontsize=7)
    ax.grid(alpha=0.4)
    plt.tight_layout()
    plt.show()


# Demonstrate: build the boustrophedon seed and visualise
sweep_s = make_sweep_synrm(15)
mask_s  = build_topology_synrm(sweep_s)
plot_polar(mask_s, 'SynRM boustrophedon seed')

print(f"Sweep deposited {int(mask_s.sum())} iron cells out of 25.")
print(f"Budget K1={K1_S}, K2={K2_S}: {'IN' if K1_S <= int(mask_s.sum()) <= K2_S else 'OUT OF'} range.")

### 5.7.2 GA configuration for SynRM

Hyper-parameters are tightened relative to the C-core because each FEMM evaluation is $\sim 6 \times$ more expensive (rotor-angle sweep) and the search space is eleven orders of magnitude smaller. Defaults from `ga_synrm.py --help`:

| Parameter | Value | Notes |
|---|---|---|
| Grid size | 5×5 (default) | configurable via `--grid-size N` for $N \in [5, 10]$ |
| Chromosome length $m$ | 15 (= MDP `MAX_STEPS` for 5×5) | one action per Q-learning step; scales to $N^2$ when `--grid-size N` is set |
| Population size $N$ | 20 | smaller than C-core ($N = 50$): each eval is 6 FEMM solves |
| Max generations | 40 | tighter than C-core (200) |
| Crossover prob. $p_{\text{cx}}$ | 0.8 per pair | same as C-core |
| Mutation prob. $p_{\text{mut}}$ | 0.3 per individual | higher than C-core (0.2) to compensate for shorter chromosome |
| Per-gene mutation $p_{\text{ind}}$ | $\max(0.05, 1/m) \approx 0.067$ | floor of 0.05 |
| Hall-of-Fame | 3 elites | smaller than C-core (5) |
| Stall termination | 12 generations | smaller than C-core (25) |
| Iron budget $[K_1, K_2]$ | $[5, 20]$ cells (5×5) | scales to $[\max(N,5),\, \lfloor 0.8 N^2 \rfloor]$ for `--grid-size N` |
| Action bias (R / L / U / D) | 30 / 20 / 30 / 20 | sweep outward from (0,0) |
| Random seed | 42 | same as C-core |

Population seeding uses the same idea as the C-core: a deterministic **boustrophedon sweep** is always inserted as the first individual. For seq-len = 15 the sweep is `RIGHT × 4, UP, LEFT × 4, UP, RIGHT × 4, UP`, depositing 15 cells across rows $i = 0, 1, 2$ plus one cell at $i = 3$. Unlike the C-core (where gen 0 is dominated by infeasible random walks), the SynRM's smaller grid and lower $K_1 = 5$ mean **most random individuals are feasible immediately** — the production run below reports `feasible = 20/20` at gen 0.

### 5.7.3 Production run (`--pop 20 --n-gen 40 --stall 15`)

Running `python ga_synrm.py --pop 20 --n-gen 40 --stall 15` reproduces the artefacts in `designs/synrm_ga/`. Summary of that run:

| Metric | Value |
|---|---|
| Generations executed | 26 (terminated by stall criterion) |
| Total FEMM evaluations | 359 (gen-0 = 20 individuals, gens 1–25 evaluate only invalid offspring) |
| First non-zero fitness | gen 0 — the boustrophedon seed and biased-random individuals are already feasible |
| Best gen-0 fitness | $7.43$ (boustrophedon seed alone) |
| Global best fitness | $f^\star = 11.29$ at gen 10 |
| Iron-cell count of best | 12 cells (well inside $[5, 20]$ budget) |
| Plateau length | 15 generations with no improvement after gen 10 |

**Behavioural notes:**

- *gen 0 is already feasible.* Every individual in the initial population satisfies $5 \le \text{iron} \le 20$ — the 5×5 grid is small enough that even slightly-biased random walks deposit $\ge 5$ cells in 15 steps. Compare with the C-core, where 33 % of cells must be filled and gen 0 is uniformly infeasible.
- *Fast climb gen 0 → 10.* Best fitness climbs from $7.43$ (boustrophedon seed) through $8.19 \to 8.92 \to 11.08 \to 11.11 \to 11.27 \to 11.29$ as crossover combines productive subsequences from the diverse initial pop.
- *Plateau gen 10 → 25.* No further improvement; stall counter trips after 15 generations and the GA exits.

**Best topology** — the winning action sequence `[3, 2, 2, 2, 3, 0, 0, 2, 0, 0, 0, 2, 0, 1, 1]` traces a path that deposits 12 cells in **two disconnected (in the design grid, but joined through the structural M-19 hub/peri rings) regions**:

```
                j=0  j=1  j=2  j=3  j=4
   i=4 (outer)   .    .    X    X    X    ← outer-band wedge (3 cells)
   i=3           X    .    X    X    X    ← (3,0) + outer-row continuation
   i=2           X    X    X    .    .    ← bridges j=0 column to corner
   i=1           X    .    .    .    .
   i=0 (inner)   X    .    .    .    .    ← inner end of the j=0 rib
```

Two structural ideas the GA discovered:
1. **Partial rib at $j = 0$** (rows 0–3, NOT row 4) — the q-axis-side angular sector. The smoke-test L-shape (Section 5.7.4 below) had a *full* rib including (4,0); the production run found that dropping the (4,0) corner and reinvesting that cell elsewhere is a net win.
2. **An outer-corner block** at rows 2–4 × cols 2–4 (a stepped wedge near the 36°–90° angular range, adjacent to the air gap). This is where the d-axis sits (geometric 45° = j = 2) plus the upper-q-axis side (cols 3–4 near $\theta = 90°$).

```{figure} designs/synrm_ga/ga_results.png
---
name: fig-synrm-ga-prod
width: 95%
---
**Production run** (`--pop 20 --n-gen 40 --stall 15`). Left: convergence curve — climb from boustrophedon seed (7.43) through gen 10 (11.29), then 15-generation plateau until stall termination. Right: best topology in polar coordinates with the geometric d-axis at 45° mech marked. Dark cells are GA-placed iron; the topology has a partial j=0 rib + an outer-corner wedge straddling the geometric d-axis.
```

**Cross-check against the diagnostic baselines** (from `designs/synrm_diag/synrm_diag_results.json`, Section 6 of the chapter):

| Topology | iron | fitness (max $\lvert t \rvert$) | comment |
|---|---:|---:|---|
| **GA production-run best** | **12** | **11.29** | partial j=0 rib + outer-corner wedge |
| Full iron (B in diagnostic) | 25 | 9.72 | GA beats full-fill with 48 % of the iron |
| Conventional 2-barrier (D) | 15 | 9.04 | 25 % more iron, 20 % lower fitness |
| GA smoke-test (Section 5.7.4) | 9 | 9.65 | naïve L-shape; 17 % below the production optimum |
| Single rib at $j = 2$ / d-axis (F) | 5 | 3.62 | textbook d-axis placement insufficient on its own |
| Empty rotor (A) | 0 | 2.74 | stator-driven baseline floor |

Three observations carry forward from the Section-6 diagnostic and the GA result reaffirms them:

1. The MMF peak from the $q = 2$ distributed winding does *not* fall on the geometric d-axis at 45° mech; both the GA smoke-test (Section 5.7.4) and the production run place significant iron at $j = 0$ (the q-axis-side angular sector) rather than $j = 2$.
2. The outer radial row $i = 4$ — adjacent to the air gap — contributes more reward per cell than any inner row. The GA's outer-corner wedge sits entirely in rows 2–4.
3. **Well-placed iron beats more iron**: 12 cells outperform 25 cells of full-iron by 16 %.

### 5.7.4 Smoke-test reference (for comparison)

For reference, the quick smoke run `--pop 10 --n-gen 10 --stall 5` (used during development) converged in 7 generations to fitness 9.65 with 9 iron cells forming a simple L-shape (full rib at $j = 0$ + full outer band at $i = 4$). Both the smoke-test and the production run place iron at $j = 0$ and on the outer row, but the production run found a richer topology — adding angular cells in the upper-right corner and dropping the (4,0) intersection of the L — that gains 17 % fitness for 33 % more iron.

### 5.7.5 Customisable grid size

`ga_synrm.py` now accepts `--grid-size N` for any $N \in [5, 10]$. The env reconfigures derived constants (`DR`, `DTHETA`), iron budget (`K1 = max(N, 5)`, `K2 = \lfloor 0.8 N^2 \rfloor`), and per-grid `.fem` cache filenames automatically. Each grid size writes to its own output directory (`designs/synrm_ga_NxN/`).

```bash
python ga_synrm.py --grid-size 7              # 7×7 polar grid -> designs/synrm_ga_7x7/
python ga_synrm.py --grid-size 10 --pop 15    # 10×10, smaller pop for tractable runtime
```

The same `--grid-size` flag is available on the matching Q-learning script (`q_learn_synrm.py`, Chapter 6).

> **Status.** Only the default $N = 5$ sweep has been executed in this jupyter-book run; the artefacts in `designs/synrm_ga/` are that default run. The $N \in \{6, 7, 8, 10\}$ sweep across `designs/synrm_ga_NxN/` is identified as planned future work in Section 7.6.2 — when those runs land, a comparative plot (best fitness vs grid resolution) will be added below this section.

In [ ]:
# Visualise the SynRM GA run from saved artifacts in designs/synrm_ga/
import json
import numpy as np
import matplotlib.pyplot as plt

lb        = json.load(open('designs/synrm_ga/ga_logbook.json'))
best_mask = np.load('designs/synrm_ga/ga_best_mask.npy')

gens   = [r['gen']  for r in lb]
maxfit = [r['max']  for r in lb]
mfit   = [r['mean'] for r in lb]
mnfit  = [r['min']  for r in lb]

fig     = plt.figure(figsize=(14, 5.5))
ax_conv = fig.add_subplot(1, 2, 1)
ax_topo = fig.add_subplot(1, 2, 2, projection='polar')

# Convergence
ax_conv.plot(gens, maxfit, 'crimson',   lw=2,   marker='o', markersize=5, label='Best')
ax_conv.plot(gens, mfit,   'steelblue', lw=1.5, marker='o', markersize=4, label='Mean')
ax_conv.fill_between(gens, mnfit, maxfit, alpha=0.15, color='steelblue')
ax_conv.set_xlabel('Generation')
ax_conv.set_ylabel('Fitness  (max |torque| across 6 rotor angles)')
ax_conv.set_title(f'SynRM GA convergence ({len(lb)} generations)')
ax_conv.legend(); ax_conv.grid(alpha=0.3)

# Polar topology
ax_topo.set_theta_zero_location('E'); ax_topo.set_theta_direction(1)
ax_topo.set_thetamin(0); ax_topo.set_thetamax(90)
ax_topo.set_ylim(0, R_PERI_IN + 4)
n_arc = 16
for i in range(DD_ROWS_S):
    for j in range(DD_COLS_S):
        r1 = R_HUB + i * DR_S
        r2 = R_HUB + (i + 1) * DR_S
        t1 = np.radians(j * DTHETA_S)
        t2 = np.radians((j + 1) * DTHETA_S)
        theta_arc  = np.linspace(t1, t2, n_arc)
        theta_poly = np.concatenate([theta_arc, theta_arc[::-1]])
        r_poly     = np.concatenate([np.full(n_arc, r1), np.full(n_arc, r2)])
        color = '#2C3E50' if best_mask[i, j] else '#FEF9E7'
        ax_topo.fill(theta_poly, r_poly, color=color, edgecolor='#7F8C8D', linewidth=0.4)
ax_topo.plot([np.radians(45)] * 2, [0, R_PERI_IN + 2], 'r--', lw=1.0, alpha=0.7)
ax_topo.text(np.radians(45), R_PERI_IN + 3.5, 'd-axis (geometric)',
             ha='center', va='bottom', fontsize=8, color='red')
ax_topo.set_title(f'GA best — {int(best_mask.sum())} iron cells, fitness {max(maxfit):.2f}', pad=18)
ax_topo.set_yticks([R_HUB, R_PERI_IN])
ax_topo.set_yticklabels([f'{R_HUB:.1f}', f'{R_PERI_IN:.1f}'], fontsize=7)
ax_topo.grid(alpha=0.4)

plt.suptitle('SynRM SeqTO-v1 — GA Result (DEAP)', fontsize=12)
plt.tight_layout()
plt.show()

best_gen = gens[maxfit.index(max(maxfit))]
print(f"Best fitness  : {max(maxfit):.4f}  at gen {best_gen}")
print(f"Iron cells    : {int(best_mask.sum())} / 25 designable  (budget {K1_S}..{K2_S})")
print(f"Total gens    : {len(lb)}  (stall-terminated)")
print(f"Total FEMM ev.: {sum(r['nevals'] for r in lb)}")

## 5.8 Summary

Genetic algorithms applied to the SeqTO-v1 chromosome formulation:

1. Operate on action sequences of length $m \ll n$ — an **8.5× reduction** for C-core ($m = 34$ vs $n = 288$) and a **1.7× reduction** for SynRM ($m = 15$ vs $n = 25$ at the default grid). Connectivity is guaranteed by the controller mechanics, eliminating the connectivity-penalty term required by ON/OFF GA formulations.
2. Use the *same* FEMM reward function consumed by the Q-learning agent — `seqTO_env.calculate_reward` for C-core, `seqTO_SynRM_env.calculate_reward` for SynRM — so the GA-vs-TD comparison is on identical objective landscapes.
3. Hard-reject out-of-budget individuals in pure Python **before** invoking FEMM, saving evaluation budget on infeasible topologies.
4. Converge on both machines under stall termination:
   - **C-core**: 48 generations / 1,808 FEMM calls, best fitness $f^\star = 1{,}628.61$ at gen 22 with 98 iron cells (Section 5.6).
   - **SynRM**: 26 generations / 359 FEMM calls, best fitness $f^\star = 11.29$ at gen 10 with 12 iron cells — a partial $j = 0$ rib plus an outer-corner wedge (Section 5.7.3). The result beats a full-iron rotor by 16 % using under half the iron.
5. Demonstrate the **algorithm-agnostic generality** of SeqTO-v1: a single chromosome encoding accommodates a planar $18 \times 35$ Cartesian actuator and a polar $5 \times 5$ quarter-symmetric motor without changing the GA itself — only the topology decoder.
6. **Customisable grid**: `ga_synrm.py --grid-size N` makes the SynRM design domain configurable for any $N \in [5, 10]$. The env's `configure()` function rescales `DR`, `DTHETA`, the iron budget, and per-grid `.fem` cache filenames; each grid size writes to its own `designs/synrm_ga_NxN/` output directory. Only $N = 5$ has been swept in this run — the $N \in \{6, 7, 8, 10\}$ sweep is planned future work (Section 7.6.2).

Force-in-Newtons benchmarking of the GA-best C-core topology, force-vs-torque benchmarking of the GA-best SynRM topology, and the side-by-side comparison with the Q-learning solutions on both machines, are deferred to a later analysis pass (Section 7 of the chapter).